In [0]:
CREATE OR REPLACE VIEW gold.analistas.vw_producaocontagem2026 AS (

WITH ESTOQUE_LOCALIZACAO AS (
    SELECT
        CodProduto,
        MAX(CodLocal) AS CodLocalEstoque,
        MAX(NomeLocal) AS DescricaoLocalEstoque
    FROM gold.sankhya.fato_estoque
    GROUP BY CodProduto
),

CAPACIDADE_E_METAS AS (
    WITH CalendarioDiasUteis AS (
        SELECT 2026 AS Ano, 1 AS Mes_Numero, 21 AS Dias_Uteis UNION ALL
        SELECT 2026, 2, 18 UNION ALL SELECT 2026, 3, 22 UNION ALL SELECT 2026, 4, 20 UNION ALL
        SELECT 2026, 5, 20 UNION ALL SELECT 2026, 6, 21 UNION ALL SELECT 2026, 7, 23 UNION ALL
        SELECT 2026, 8, 21 UNION ALL SELECT 2026, 9, 21 UNION ALL SELECT 2026, 10, 21 UNION ALL
        SELECT 2026, 11, 19 UNION ALL SELECT 2026, 12, 21
    ),
    TETO_FAMILIA AS (
        SELECT 2026 AS Ano, 'MONITORES' AS Familia, 6500 AS Capacidade_Mensal_Ajustada, 0.98 AS Meta_Capacidade UNION ALL
        SELECT 2026, 'DESKTOP', 2790, 0.85 UNION ALL
        SELECT 2026, 'AIO', 500, 0.80
    )
    SELECT
        c.Ano,
        c.Mes_Numero,
        t.Familia,
        t.Familia AS Subgrupo,
        CAST(NULL AS INT) AS QDE_Fixa,
        CAST(NULL AS DOUBLE) AS Cap_Hora,
        t.Capacidade_Mensal_Ajustada AS Cap_Turno_8h,
        t.Meta_Capacidade,
        c.Dias_Uteis,
        ROUND(t.Capacidade_Mensal_Ajustada / CAST(c.Dias_Uteis AS DOUBLE), 4) AS Dias_Alocados_Mix,
        t.Capacidade_Mensal_Ajustada
    FROM TETO_FAMILIA t
    INNER JOIN CalendarioDiasUteis c ON t.Ano = c.Ano
),

ULTIMA_ATIVIDADE_OP_2026 AS (
    SELECT
        OrdemProducao,
        MAX(CodItemAtividade) AS CodItemAtividade_Final
    FROM gold.sankhya.fato_atividade_op
    GROUP BY OrdemProducao
),

BASE_PRODUCAO_SLA_2026 AS (
    -- Grão: (OP, Produto, Data) — cada dia de apontamento vira uma linha própria
    -- Corrige divergência vs PCP que conta pela data real de cada bipagem
    SELECT
        act.OrdemProducao,
        app.CodProdutoAcabado                           AS CodProd,
        YEAR(CAST(ap.DataHoraApontamento AS DATE))      AS Ano,
        MONTH(CAST(ap.DataHoraApontamento AS DATE))     AS Mes,
        WEEKOFYEAR(CAST(ap.DataHoraApontamento AS DATE)) AS Semana,
        CAST(ap.DataHoraApontamento AS DATE)            AS Data_Producao,
        MIN(ap.DataHoraApontamento)                     AS DataHora_Apontamento,
        SUM(CAST(app.QuantidadeApontada AS INT))        AS Quantidade_Produzida
    FROM gold.sankhya.fato_apontamento ap
    INNER JOIN ULTIMA_ATIVIDADE_OP_2026 ult ON ap.CodItemAtividade = ult.CodItemAtividade_Final
    INNER JOIN gold.sankhya.fato_atividade_op act ON ap.CodItemAtividade = act.CodItemAtividade
    INNER JOIN gold.sankhya.fato_apontamento_produto app ON ap.CodApontamentoUnico = app.CodApontamentoUnico
    WHERE ap.DataHoraApontamento IS NOT NULL
    GROUP BY act.OrdemProducao, app.CodProdutoAcabado, CAST(ap.DataHoraApontamento AS DATE)
),

SKU_ATRIBUTOS_2026 AS (
    SELECT
        P.CodProduto, P.DescricaoProduto, P.Marca AS Fornecedor, P.ModeloMkt AS Modelo,
        GP.NomeGrupoPai AS Familia, GP.LinhaDeNegocio, GP.NomeGrupoFamilia, P.UsadoComo
    FROM gold.sankhya.dim_produtos P
    INNER JOIN gold.sankhya.dim_grupo_produtos GP ON GP.CodGrupoProduto = P.CodGrupoProduto
    WHERE P.UsadoComo IN ('Venda (fabricação própria)', 'Revenda')
      AND GP.LinhaDeNegocio IN ('WordPC/Skill')
      AND GP.NomeGrupoPai <> 'COMPONENTES'
),

PLANEJAMENTO_OP_2026 AS (
    SELECT OrdemProducao, CodProdutoAcabado, MAX(CAST(QuantidadeAProduzir AS INT)) AS Qtd_Planejada_Item
    FROM gold.sankhya.fato_ordem_producao_item
    GROUP BY OrdemProducao, CodProdutoAcabado
),

LINK_LOGISTICO_2026 AS (
    SELECT OrdemProducao, MAX(NotaFaturamento) AS NumUnicoNota
    FROM gold.sankhya.fato_instancia_item_nota
    GROUP BY OrdemProducao
),

BIPAGENS_PIVOT_2026 AS (
    SELECT
        fao.OrdemProducao,
        fap.CodProdutoAcabado,
        SUM(CASE WHEN pt.DescricaoPosto LIKE '%1%' OR pt.DescricaoPosto LIKE '%Prepara%' THEN CAST(fap.QuantidadeApontada AS INT) ELSE 0 END) AS Bips_Separacao,
        SUM(CASE WHEN pt.DescricaoPosto LIKE '%2%' OR pt.DescricaoPosto LIKE '%Conferen%' THEN CAST(fap.QuantidadeApontada AS INT) ELSE 0 END) AS Bips_Montagem,
        SUM(CASE WHEN pt.DescricaoPosto LIKE '%3%' OR pt.DescricaoPosto LIKE '%Qualidade%' THEN CAST(fap.QuantidadeApontada AS INT) ELSE 0 END) AS Bips_Qualidade,
        SUM(CASE WHEN pt.DescricaoPosto LIKE '%4%' OR pt.DescricaoPosto LIKE '%Runin%' OR pt.DescricaoPosto LIKE '%Run-in%' THEN CAST(fap.QuantidadeApontada AS INT) ELSE 0 END) AS Bips_Runin,
        SUM(CASE WHEN pt.DescricaoPosto LIKE '%5%' OR pt.DescricaoPosto LIKE '%Embala%' THEN CAST(fap.QuantidadeApontada AS INT) ELSE 0 END) AS Bips_Embalagem
    FROM gold.sankhya.fato_atividade_op fao
    INNER JOIN gold.sankhya.dim_posto_trabalho pt ON fao.CodCentroTrabalho = pt.CodCentroTrabalho
    INNER JOIN gold.sankhya.fato_apontamento fa ON fao.CodItemAtividade = fa.CodItemAtividade
    INNER JOIN gold.sankhya.fato_apontamento_produto fap ON fa.CodApontamentoUnico = fap.CodApontamentoUnico
    WHERE fap.QuantidadeApontada > 0
    GROUP BY fao.OrdemProducao, fap.CodProdutoAcabado
),

REPAROS_CONSOLIDADO_2026 AS (
    SELECT
        cab.OrdemProducao,
        cab.CodProdutoAcabado,
        COUNT(DISTINCT cab.CodLancamentoReparo) AS Qtd_Eventos_Reparo,
        SUM(CAST(item.QuantidadeNegociada AS INT)) AS Total_Pecas_Reparadas,
        CONCAT_WS(', ', COLLECT_SET(sint.DescricaoSintoma)) AS Lista_Sintomas,
        CONCAT_WS(', ', COLLECT_SET(def.DescricaoDefeito)) AS Lista_Defeitos,
        CONCAT_WS(', ', COLLECT_SET(acao.DescricaoAcaoCorretiva)) AS Lista_Acoes,
        CONCAT_WS(', ', COLLECT_SET(DISTINCT CAST(item.CodMateriaPrima AS STRING))) AS SKUs_Pecas_Trocadas,
        CONCAT_WS(' | ', COLLECT_SET(DISTINCT prod_peca.DescricaoProduto)) AS Descricao_Pecas_Trocadas,
        CONCAT_WS(', ', COLLECT_SET(DISTINCT CAST(CAST(item.DataHoraFimReparo AS DATE) AS STRING))) AS Datas_Fim_Reparo,
        CONCAT_WS(', ', COLLECT_SET(DISTINCT COALESCE(cab.SerieProdutoAcabado, 'S/N'))) AS Series_PA_Reparados,
        CONCAT_WS(' | ', COLLECT_SET(DISTINCT COALESCE(pt_rep.DescricaoPosto, CAST(cab.CodCentroTrabalho AS STRING)))) AS Centros_Trabalho_Reparo
    FROM gold.sankhya.fato_producao_reparo_cabecalho cab
    INNER JOIN gold.sankhya.fato_producao_reparo_item item ON cab.CodLancamentoReparo = item.CodLancamentoReparo
    LEFT JOIN gold.sankhya.dim_produtos prod_peca ON item.CodMateriaPrima = prod_peca.CodProduto
    LEFT JOIN gold.sankhya.dim_posto_trabalho pt_rep ON cab.CodCentroTrabalho = pt_rep.CodCentroTrabalho
    LEFT JOIN gold.sankhya.dim_producao_sintoma_reparo sint ON cab.CodSintoma = sint.CodSintoma
    LEFT JOIN gold.sankhya.dim_producao_defeito_reparo def ON item.CodDefeito = def.CodDefeito
    LEFT JOIN gold.sankhya.dim_producao_acao_reparo acao ON item.CodAcaoCorretiva = acao.CodAcaoCorretiva
    WHERE cab.StatusReparo = 'Concluído'
    GROUP BY cab.OrdemProducao, cab.CodProdutoAcabado
)

SELECT
    EST.CodLocalEstoque,
    EST.DescricaoLocalEstoque,
    COALESCE(METAS.Meta_Capacidade, 0.95)            AS Meta_Capacidade,
    CASE WHEN ROW_NUMBER() OVER (PARTITION BY CONSULTA.Ano, CONSULTA.Mes, METAS.Familia ORDER BY CONSULTA.DataProducao, CONSULTA.OP) = 1
         THEN COALESCE(METAS.Capacidade_Mensal_Ajustada, 0) ELSE 0 END AS Cap_Mensal_Ajustada_Subgrupo,
    CASE WHEN ROW_NUMBER() OVER (PARTITION BY CONSULTA.Ano, CONSULTA.Mes, METAS.Familia ORDER BY CONSULTA.DataProducao, CONSULTA.OP) = 1
         THEN COALESCE(METAS.Cap_Turno_8h, 0) ELSE 0 END              AS Cap_Turno_8h_Subgrupo,
    COALESCE(METAS.Dias_Uteis, 0)                 AS Dias_Uteis_Mes,
    CONSULTA.*
FROM (
    SELECT
        'CONTAGEM/MG'                                               AS Planta,
        pp.DescricaoProcesso                                        AS Processo,
        PROD.OrdemProducao                                          AS OP,
        LINK.NumUnicoNota                                           AS NumUnicoNota,
        O.NumNota                                                   AS NumeroNota,
        PROD.Ano, PROD.Mes, PROD.Semana, PROD.Data_Producao        AS DataProducao,
        PROD.DataHora_Apontamento                                   AS DataHoraApontamento,
        
        CASE MONTH(PROD.DataHora_Apontamento)
            WHEN 1  THEN '01. JANEIRO' WHEN 2  THEN '02. FEVEREIRO' WHEN 3  THEN '03. MARÇO'
            WHEN 4  THEN '04. ABRIL'   WHEN 5  THEN '05. MAIO'      WHEN 6  THEN '06. JUNHO'
            WHEN 7  THEN '07. JULHO'   WHEN 8  THEN '08. AGOSTO'    WHEN 9  THEN '09. SETEMBRO'
            WHEN 10 THEN '10. OUTUBRO' WHEN 11 THEN '11. NOVEMBRO'  WHEN 12 THEN '12. DEZEMBRO'
        END AS Mes_Ordenado,

        PROD.CodProd, SKU.DescricaoProduto, SKU.Fornecedor, SKU.Modelo,
        SKU.Familia, SKU.LinhaDeNegocio, SKU.NomeGrupoFamilia, SKU.UsadoComo,

        CASE
            WHEN UPPER(SKU.Familia) IN ('MONITORES', 'MONITOR') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '49("| |POL|INCH)' THEN '49"'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(32|34)("| |POL|INCH)' THEN REGEXP_EXTRACT(UPPER(SKU.DescricaoProduto), '(32|34)', 1) || '"'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '27("| |POL|INCH)' THEN '27"'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(19|19\.5|20|21|21\.5|22|23\.8|24)("| |POL|INCH)?' THEN REGEXP_EXTRACT(UPPER(SKU.DescricaoProduto), '(19\.5|21\.5|23\.8|19|20|21|22|24)', 1) || '"'
                    ELSE 'Indefinida'
                END
            WHEN UPPER(SKU.Familia) IN ('TVS', 'TV', 'TELEVISORES') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(24|32|43|50|55|60|65|70|75)("| |POL|INCH)?' THEN REGEXP_EXTRACT(UPPER(SKU.DescricaoProduto), '(24|32|43|50|55|60|65|70|75)', 1) || '"'
                    ELSE 'Indefinida'
                END
            ELSE 'N/A'
        END AS Polegada_Estimada,

        CASE
            WHEN UPPER(SKU.Familia) IN ('MONITORES', 'MONITOR') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '49("| |POL|INCH)' THEN 'M49'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(32|34)("| |POL|INCH)' THEN 'M32/34'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '27("| |POL|INCH)' THEN 'M27'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(19|19\.5|20|21|21\.5|22|23\.8|24)("| |POL|INCH)?' THEN 'M19/21/24'
                    ELSE 'MONITOR - VERIFICAR CADASTRO'
                END
            WHEN UPPER(SKU.Familia) IN ('TVS', 'TV', 'TELEVISORES') THEN
                CASE
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(24|32)("| |POL|INCH)?' THEN 'TV24/32'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(43|50)("| |POL|INCH)?' THEN 'TV43/50'
                    WHEN UPPER(SKU.DescricaoProduto) RLIKE '(55|60|65|70|75)("| |POL|INCH)?' THEN 'TV60/65'
                    ELSE 'TV - VERIFICAR CADASTRO'
                END
            ELSE 'OUTROS'
        END AS Subgrupo_TV_Monitor,

        CASE
            WHEN UPPER(SKU.DescricaoProduto) RLIKE '(AIO|ALL IN ONE|ALL-IN-ONE)'
                 OR CAST(SKU.CodProduto AS STRING) IN (
                        '96511', '96513', '96514', '96515', '96516', '96518', '96519', '96520', '96521', '96523', '96524', '96525', '96526', '96527', '96528', '96530', '110389', '110390', '110391', '110392', '110393', '110394', '110395', '110398', '110399', '110404', '110405', '110406', '110408', '110409', '110410', '110411', '110414', '110416', '110417', '110418', '110419', '110420', '110421', '110423', '110424', '110425', '110426', '112199', '117263', '131187', '131188', '131366', '131369', '131371', '131372', '131374', '131376', '131377', '131379', '131381', '131382', '131383', '131384', '131385', '131386', '89093', '130974', '131497'
                 ) THEN 'AIO'
            WHEN UPPER(SKU.Familia) IN ('DESKTOP', 'DESKTOPS', 'COMPUTADORES') OR UPPER(SKU.Familia) LIKE '%DESKTOP%' THEN
                CASE
                    WHEN CAST(SKU.CodProduto AS STRING) IN (
                       '72326', '72327', '72328', '72329', '72331', '72351', '72353', '72357', '72861', '72869', '72881', '77323', '77371', '77372', '77373', '77374', '77375', '77377', '77378', '77379', '77380', '77381', '77382', '77383', '77384', '77386', '77387', '77388', '77390', '77392', '77394', '80648', '80650', '80652', '80653', '80654', '80655', '80656', '80657', '80658', '80659', '80660', '80661', '80821', '80825', '80826', '80827', '80828', '80829', '80830', '80831', '80833', '80835', '80836', '85225', '85227', '85228', '85229', '85230', '85236', '85250', '85251', '85256', '85257', '85269', '85271', '85273', '85279', '85280', '85284', '85286', '85287', '85288', '85289', '85292', '85293', '85294', '85305', '85307', '85308', '85309', '85310', '85312', '85314', '85320', '85329', '85331', '89114', '89115', '89117', '89118', '89121', '89124', '89125', '89127', '89130', '89131', '89132', '91084', '91091', '91111', '92608', '92611', '92613', '92614', '92617', '92618', '92619', '92620', '92622', '92623', '92624', '92626', '92627', '92628', '92629', '92630', '92631', '92632', '92633', '92634', '92635', '92636', '92637', '92638', '93757', '93758', '93759', '93760', '93761', '93762', '93763', '93764', '93765', '93766', '93768', '93769', '93770', '93771', '93772', '93773', '93774', '93775', '93776', '93777', '93778', '93779', '93800', '93801', '93802', '93803', '93805', '93808', '93811', '93815', '93818', '93819', '93824', '93831', '93832', '93833', '93834', '104483', '104487', '104489', '104492', '104493', '104494', '104497', '104499', '104506', '104507', '104509', '104511', '104513', '104515', '104525', '104526', '104533', '104534', '104537', '104540', '104541', '104542', '104544', '104545', '104547', '104553', '104554', '104555', '104556', '104561', '104564', '104567', '104570', '104573', '104575', '104576', '104577', '104578', '104607', '104609', '104610', '104612', '104614', '104615', '104619', '104620', '104622', '104623', '104624', '104625', '104626', '104627', '104629', '104630', '104631', '104633', '104634', '104636', '104637', '104639', '104640', '104642', '104643', '104644', '104645', '104647', '104648', '104649', '104650', '104656', '104658', '104659', '104660', '104661', '104662', '104663', '104667', '104668', '104669', '104671', '106249', '106250', '106251', '106253', '106254', '106255', '106256', '106257', '106258', '106259', '106260', '106261', '106262', '106264', '106265', '106266', '106267', '106268', '106269', '106270', '106271', '106272', '106273', '106283', '106616', '106618', '106619', '106620', '106621', '106622', '106623', '106624', '131388', '131391', '131393', '131394', '131397', '131399', '131401', '131404', '131408', '131409', '131410', '131411', '131412', '131413', '131414', '131415', '131416', '131417', '131418', '131419', '131420', '131421', '131422', '131423', '91112', '91113', '91114', '91115', '91117', '91120', '91121', '91122', '91154', '91157', '91159', '91165', '132041', '132042', '132043', '132044', '132045', '132046', '132047', '132048', '132049', '132050', '132051', '132052', '132053', '132054'
                    ) THEN 'GAMER'
                    WHEN CAST(SKU.CodProduto AS STRING) IN (
                       '72473', '72479', '72480', '72481', '72482', '72483', '72484', '72490', '72491', '72492', '72523', '72524', '72528', '72531', '72532', '72533', '72534', '72542', '72560', '72561', '72562', '72564', '72571', '72572', '72576', '73483', '73484', '73485', '73486', '73513', '73514', '73515', '73516', '73544', '73546', '73548', '73549', '90805', '90806', '90808', '90810', '90811', '90812', '90813', '90814', '90815', '90929', '90931', '90933', '90934', '90937', '90938', '90939', '90944', '90946', '90947', '98433', '98434', '98435', '98437', '98438', '98443', '98444', '98445', '98448', '98450', '98452', '98453', '98454', '98457', '98458', '98459', '98460', '98462', '98464', '98465', '98466', '98468', '98469', '98470', '98471', '98473', '98474', '98475', '98476', '98483', '98484', '98485', '98486', '98487', '98488', '98489', '98490', '98492', '98493', '98494', '98495', '98496', '98497', '98499', '98500', '98501', '98502', '98503', '98505', '98506', '98507', '98508', '98509', '98510', '98511', '98513', '98526', '98527', '98528', '98529', '98530', '98531', '98532', '98534', '98535', '98536', '98537', '98538', '98539', '98540', '98541', '98542'
                    ) THEN 'DESK'
                    ELSE
                        CASE
                            WHEN UPPER(SKU.DescricaoProduto) LIKE '%GAMER%' THEN 'GAMER'
                            WHEN UPPER(SKU.DescricaoProduto) RLIKE '(RTX|RX ?5[87]0|RX ?6[0-9]|RX ?7[0-9]|GTX ?16|GTX ?10[67]0)' THEN 'GAMER'
                            WHEN UPPER(SKU.DescricaoProduto) LIKE '%SLIM%' THEN 'SLIM'
                            ELSE 'DESK'
                        END
                END
            ELSE 'OUTROS'
        END AS SubgrupoProduto,

        PROD.Quantidade_Produzida                                   AS Qtd_Produzida,
        -- Planejado proporcional: distribui o total planejado da OP conforme o peso de cada dia na produção total
        -- Garante OFR mensal ≤ 100% quando não há superprodução real no grão OP
        CAST(ROUND(
            COALESCE(PLAN_OP.Qtd_Planejada_Item, 0)
            * CAST(PROD.Quantidade_Produzida AS DOUBLE)
            / NULLIF(SUM(PROD.Quantidade_Produzida) OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd), 0)
        , 0) AS INT) AS Qtd_Planejada,
        -- Saldo permanece no grão OP (primeira linha) — mostra pendência total
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(PLAN_OP.Qtd_Planejada_Item, 0) - SUM(PROD.Quantidade_Produzida) OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd)
             ELSE 0 END AS Saldo,
        'PRODUZIDO'                                                 AS Situacao,
        CAST(O.DataNegociacao AS DATE)                              AS DataFaturamento,
        
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(BIP.Bips_Separacao, 0) ELSE 0 END AS Separacao,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(BIP.Bips_Montagem, 0) ELSE 0 END  AS Montagem,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(BIP.Bips_Qualidade, 0) ELSE 0 END AS Qualidade,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(BIP.Bips_Runin, 0) ELSE 0 END     AS Runin,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(BIP.Bips_Embalagem, 0) ELSE 0 END AS Embalagem,
        
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(REP.Qtd_Eventos_Reparo, 0) ELSE 0 END    AS Qtd_Reparos,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(REP.Total_Pecas_Reparadas, 0) ELSE 0 END AS Pecas_Substituidas,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(NULLIF(REP.Lista_Sintomas, ''), 'Sem Registro') ELSE 'Sem Registro' END AS Sintomas,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(NULLIF(REP.Lista_Defeitos, ''), 'Sem Registro') ELSE 'Sem Registro' END AS Defeitos,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(NULLIF(REP.Lista_Acoes, ''), 'Sem Registro') ELSE 'Sem Registro' END    AS Acoes,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(REP.SKUs_Pecas_Trocadas, 'Nenhuma') ELSE 'Nenhuma' END     AS SKU_Componente_Trocado,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(REP.Descricao_Pecas_Trocadas, 'Nenhuma') ELSE 'Nenhuma' END AS Descricao_Componente_Trocado,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(REP.Datas_Fim_Reparo, 'Nenhum') ELSE 'Nenhum' END          AS Datas_Conclusao_Reparo,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(REP.Series_PA_Reparados, 'Nenhum') ELSE 'Nenhum' END       AS Serie_PA,
        CASE WHEN ROW_NUMBER() OVER (PARTITION BY PROD.OrdemProducao, PROD.CodProd ORDER BY PROD.Data_Producao) = 1
             THEN COALESCE(REP.Centros_Trabalho_Reparo, 'Nenhum') ELSE 'Nenhum' END   AS Centro_Trabalho_Reparo
    FROM BASE_PRODUCAO_SLA_2026 PROD
    INNER JOIN SKU_ATRIBUTOS_2026 SKU ON PROD.CodProd = SKU.CodProduto
    INNER JOIN gold.sankhya.fato_ordem_producao fop ON PROD.OrdemProducao = fop.OrdemProducao
    INNER JOIN gold.sankhya.dim_processo_producao pp ON fop.CodProcessoUnico = pp.CodProcessoUnico
    LEFT JOIN PLANEJAMENTO_OP_2026 PLAN_OP ON PROD.OrdemProducao = PLAN_OP.OrdemProducao AND PROD.CodProd = PLAN_OP.CodProdutoAcabado
    LEFT JOIN LINK_LOGISTICO_2026 LINK ON PROD.OrdemProducao = LINK.OrdemProducao
    LEFT JOIN gold.sankhya.fato_operacoes O ON LINK.NumUnicoNota = O.NumUnicoNota
    LEFT JOIN BIPAGENS_PIVOT_2026 BIP ON PROD.OrdemProducao = BIP.OrdemProducao AND PROD.CodProd = BIP.CodProdutoAcabado
    LEFT JOIN REPAROS_CONSOLIDADO_2026 REP ON PROD.OrdemProducao = REP.OrdemProducao AND PROD.CodProd = REP.CodProdutoAcabado
    WHERE PROD.Data_Producao >= '2026-01-01' AND PROD.Data_Producao <= CURRENT_DATE()
      AND pp.DescricaoProcesso IN ('2 - COMPUTADORES MAESTRO PLUS', '6 - MONITOR MAESTRO PLUS')
) AS CONSULTA
LEFT JOIN ESTOQUE_LOCALIZACAO EST ON CONSULTA.CodProd = EST.CodProduto
LEFT JOIN CAPACIDADE_E_METAS METAS ON CONSULTA.Ano = METAS.Ano AND CONSULTA.Mes = METAS.Mes_Numero AND (
    CASE
        WHEN CONSULTA.SubgrupoProduto = 'AIO' THEN 'AIO'
        WHEN CONSULTA.Subgrupo_TV_Monitor IN ('M19/21/24', 'M27', 'M32/34', 'M49') THEN 'MONITORES'
        WHEN CONSULTA.SubgrupoProduto IN ('DESK', 'GAMER', 'SLIM') THEN 'DESKTOP'
        ELSE NULL
    END
) = METAS.Familia
);